# Augmenting AI Agents with Persistent Memory

## Part 1: Setting Up Database, Vector Stores and Embedding Models

This notebook explores how AI agents can be augmented with persistent memory to store, retrieve, and apply knowledge across interactions.

Large Language Model (LLM) calls are inherently stateless. Without an external memory system, an LLM cannot retain and recall information beyond a single interaction.

Memory enables agents to retain information across interactions, retrieve relevant context efficiently, and make more informed decisions during execution.

### Why Memory Matters

Persistent memory allows agents to:

- Maintain context awareness across conversations and sessions.
- Perform long-horizon tasks that span multiple interactions.
- Adapt their behavior based on past experiences and user preferences.
- Reduce redundant queries, processing, and repeated work.

### Real-World Examples

- Educational agents that track a student's learning progress over weeks or months.
- Healthcare assistants and robots that retain patient interaction history and relevant context.

This notebook goes beyond a naïve memory implementation. Rather than simply storing raw conversation transcripts, it explores structured memory systems that organize information for efficient storage, retrieval, and reasoning.

### Project

The notebook uses an AI-powered research assistant as the reference use case. The assistant helps users investigate complex topics across multiple sessions by maintaining relevant context, including previously discovered information, source metadata, and user preferences.

This persistent memory enables the agent to provide more consistent, context-aware responses while minimizing redundant research and repeated processing.

### Key Concepts Demonstrated

This implementation covers:

- Modeling different categories of agent memory and understanding their storage requirements.
- Designing a persistent memory architecture using relational and vector-based storage mechanisms.
- Implementing semantic memory using Oracle Vector Search, including embedding generation, vector indexing, and metadata filtering.
- Building a Memory Manager responsible for orchestrating memory creation, retrieval, updates, and lifecycle operations.
- Evaluating memory system design trade-offs involving retrieval quality, latency, scalability, operational cost, and reliability.

### Learning Outcomes

This notebook provides practical experience in:

- Understanding the role of memory in long-running, stateful AI agents.
- Mapping memory types to appropriate persistence and retrieval technologies.
- Implementing vector-based semantic retrieval using Oracle Database vector capabilities.
- Designing a centralized memory orchestration layer for agent applications.
- Analyzing architectural trade-offs involved in building production-ready agent memory systems.

In [ ]:
# Install dependencies
%pip install -r requirements.txt


In [ ]:
# Start Oracle AI Database
!docker compose -f ../.devcontainer/docker-compose.yml up -d oracle

In [ ]:
from helper import suppress_warnings

# Warning control
suppress_warnings()

from helper import load_env, setup_oracle_database, connect_to_oracle

load_env()

# One-time admin setup: configures tablespace, vector memory, and VECTOR user
setup_oracle_database()

# Connect as the VECTOR user for all subsequent operations
database_connection = connect_to_oracle(
    user="VECTOR",
    password="VectorPwd_2025",
    dsn="127.0.0.1:1521/FREEPDB1",
    program="devrel.deeplearning.course_1",
)

print("Using user:", database_connection.username)

### Loading the Embedding Model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

### Define Memory Tables and Stores
First, we define table names for each memory type. 

These tables will be created in Oracle Database to persist agent memory.

### Memory Types We'll Implement

| Memory Type | Human Analogy | Purpose | Storage | Retrieval Strategies Used |
|-------------|---------------|---------|---------|---------------------------|
| **Conversational** | Short-term memory | Chat history per thread | SQL Table | Exact match by thread_id |
| **Knowledge Base** | Long-term semantic memory | Facts, documents, search results | Vector Store | Semantic similarity search |
| **Workflow** | Procedural memory | Learned action patterns | Vector Store | Semantic similarity search + metadata filtering |
| **Toolbox** | Skill memory | Available tools & capabilities | Vector Store | Semantic similarity search |
| **Entity** | Episodic memory | People, places, systems mentioned | Vector Store | Semantic similarity search |
| **Summary** | Compressed memory | Condensed context for long conversations | Vector Store | Semantic similarity search (with optional ID filter) |
| **Tool Log** | Execution audit trail | Raw tool inputs/outputs and execution status | SQL Table | Exact match by thread_id + timestamp ordering |


In [ ]:
# Table names for each memory type
CONVERSATIONAL_TABLE   = "CONVERSATIONAL_MEMORY" # Episodic memory
KNOWLEDGE_BASE_TABLE   = "SEMANTIC_MEMORY" # Semantic memory
WORKFLOW_TABLE = "WORKFLOW_MEMORY" # Procedural memory
TOOLBOX_TABLE    = "TOOLBOX_MEMORY" # Procedural memory
ENTITY_TABLE = "ENTITY_MEMORY" # Semantic memory
SUMMARY_TABLE = "SUMMARY_MEMORY" # Semantic memory
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY" # Tool execution logs

ALL_TABLES = [
    CONVERSATIONAL_TABLE, 
    KNOWLEDGE_BASE_TABLE, 
    WORKFLOW_TABLE, 
    TOOLBOX_TABLE, 
    ENTITY_TABLE, 
    SUMMARY_TABLE, 
    TOOL_LOG_TABLE]

# Drop existing tables to start fresh
for table in ALL_TABLES:
    try:
        with database_connection.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
            print(f"  - {table} (dropped)")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")
            
database_connection.commit()

### Create Conversational Memory Table

This function below creates a SQL table to store chat history. 

Unlike vector stores, conversational memory uses a traditional table because we need exact retrieval by thread ID (not similarity search).

**What it does:**
- Creates a table with columns: `id`, `thread_id`, `role`, `content`, `timestamp`, `metadata`
- Adds an index on `thread_id` for fast conversation lookups
- Adds an index on `timestamp` for chronological ordering


In [ ]:
def create_conversational_history_table(conn, table_name: str = "CONVERSATIONAL_MEMORY"):
    """
    Create a table to store conversational history.

    Args:
        conn: Oracle database connection
        table_name: Name of the table to create
    """
    with conn.cursor() as cur:
        # Drop table if exists
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass  # Table doesn't exist
        
        # Create table with proper schema
        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                role VARCHAR2(50) NOT NULL,
                content CLOB NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata CLOB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                summary_id VARCHAR2(100) DEFAULT NULL
            )
        """)
        
        # Create index on thread_id for faster lookups
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_thread_id ON {table_name}(thread_id)
        """)
        
        # Create index on timestamp for ordering
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_timestamp ON {table_name}(timestamp)
        """)
        
    conn.commit()
    print(f"Table {table_name} created successfully with indexes")
    return table_name


In [ ]:
from helper import create_tool_log_table

# Create the SQL memory tables
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(database_connection, CONVERSATIONAL_TABLE)
TOOL_LOG_HISTORY_TABLE = create_tool_log_table(database_connection, TOOL_LOG_TABLE)

### Create Vector Stores for Each Memory Type

Here we create 5 separate vector stores—one for each memory type. 

Each vector store is backed by its own Oracle table and uses the same embedding model for consistency.

| Vector Store | Purpose |
|--------------|---------|
| `knowledge_base_vs` | Store documents, facts, and search results |
| `workflow_vs` | Store learned action patterns and tool sequences |
| `toolbox_vs` | Store tool definitions for semantic tool discovery |
| `entity_vs` | Store extracted entities (people, places, systems) |
| `summary_vs` | Store compressed summaries for long conversations |


In [ ]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb.retrievers.hybrid_search import (
    OracleVectorizerPreference
)


class StoreManager:
    """Manages all stores (vector stores and SQL tables) with getter methods for easy access."""
    
    def __init__(self, client, embedding_function, table_names, distance_strategy, conversational_table, tool_log_table: str | None = None):
        """
        Initialize all stores.
        
        Args:
            client: Oracle database connection
            embedding_function: Embedding model to use
            table_names: Dict with keys: knowledge_base, workflow, toolbox, entity, summary
            distance_strategy: Distance strategy for vector search
            conversational_table: Name of the conversational history SQL table
            tool_log_table: Name of the SQL tool log table
        """
        self.client = client
        self.embedding_function = embedding_function
        self.distance_strategy = distance_strategy
        self._conversational_table = conversational_table
        self._tool_log_table = tool_log_table
        
        # Initialize all vector stores
        self._knowledge_base_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['knowledge_base'],
            distance_strategy=distance_strategy,
        )
        
        self._workflow_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['workflow'],
            distance_strategy=distance_strategy,
        )
        
        self._toolbox_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['toolbox'],
            distance_strategy=distance_strategy,
        )
        
        self._entity_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['entity'],
            distance_strategy=distance_strategy,
        )
        
        self._summary_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['summary'],
            distance_strategy=distance_strategy,
        )
        
        # Store hybrid search preference for knowledge base (optional)
        self._kb_vectorizer_pref = None
    
    def get_knowledge_base_store(self):
        """Return the knowledge base vector store."""
        return self._knowledge_base_vs
    
    def get_workflow_store(self):
        """Return the workflow vector store."""
        return self._workflow_vs
    
    def get_toolbox_store(self):
        """Return the toolbox vector store."""
        return self._toolbox_vs
    
    def get_entity_store(self):
        """Return the entity vector store."""
        return self._entity_vs
    
    def get_summary_store(self):
        """Return the summary vector store."""
        return self._summary_vs
    
    def get_conversational_table(self):
        """Return the conversational history table name."""
        return self._conversational_table
    

    def get_tool_log_table(self):
        """Return the tool log table name."""
        return self._tool_log_table

    def setup_hybrid_search(self, preference_name="KB_VECTORIZER_PREF"):
        """
        Set up hybrid search for knowledge base.
        Creates vectorizer preference for hybrid indexing.
        """
        self._kb_vectorizer_pref = OracleVectorizerPreference.create_preference(
            vector_store=self._knowledge_base_vs,
            preference_name=preference_name
        )
        return self._kb_vectorizer_pref


Vector stores are wired through `StoreManager`

In [ ]:
# Create StoreManager instance
store_manager = StoreManager(
    client=database_connection,
    embedding_function=embedding_model,
    table_names={
        'knowledge_base': KNOWLEDGE_BASE_TABLE,
        'workflow': WORKFLOW_TABLE,
        'toolbox': TOOLBOX_TABLE,
        'entity': ENTITY_TABLE,
        'summary': SUMMARY_TABLE,
    },
    distance_strategy=DistanceStrategy.COSINE,
    conversational_table=CONVERSATION_HISTORY_TABLE,
    tool_log_table=TOOL_LOG_HISTORY_TABLE,
)

In [ ]:
# Get all stores via the manager
conversation_table = store_manager.get_conversational_table()
knowledge_base_vs = store_manager.get_knowledge_base_store()
workflow_vs = store_manager.get_workflow_store()
toolbox_vs = store_manager.get_toolbox_store()
entity_vs = store_manager.get_entity_store()
summary_vs = store_manager.get_summary_store()
tool_log_table = store_manager.get_tool_log_table()

In [ ]:
from helper import safe_create_index

print("Creating vector indexes...")
safe_create_index(database_connection, knowledge_base_vs, "knowledge_base_vs_ivf")
safe_create_index(database_connection, workflow_vs, "workflow_vs_ivf")
safe_create_index(database_connection, toolbox_vs, "toolbox_vs_ivf")
safe_create_index(database_connection, entity_vs, "entity_vs_ivf")
safe_create_index(database_connection, summary_vs, "summary_vs_ivf")
print("All indexes created!")

### Classification of Memory operation in Agentic Systems

A key design decision in memory engineering is determining which operations should be **Deterministic** (executed automatically by code) versus **Agent-Triggered** (decided by the LLM at runtime).

- A **deterministic** memory operation is one that runs based on system rules, not the model’s discretion. It is executed every time (or under clearly defined, non-negotiable conditions) so the system behaves predictably.
- An **agent-triggered** memory operation runs only when the model decides it’s necessary, based on intent and situation.

| Operation | Deterministic | Agent-Triggered |
|-----------|:------------:|:-------:|
| `read_conversational_memory()` | ✅ | ❌ |
| `read_knowledge_base()` | ✅ | ❌ |
| `read_workflow()` | ✅ | ❌ |
| `read_entity()` | ✅ | ❌ |
| `read_summary_context()` | ❌ | ✅ |
| `write_conversational_memory()` | ✅ | ❌ |
| `write_workflow()` | ✅ | ❌ |
| `write_entity()` | ❌ | ✅ |
| `search_tavily()` | ❌ | ✅ |
| `expand_summary()` | ❌ | ✅ |
| `summarize_and_store()` | ❌ | ✅ |
| `read_toolbox()` | ✅ | ✅ |


Deterministic memory operations run:
- **every turn**, or
- under **explicit, fixed conditions** (e.g., “always at the start of the agent loop”, “always after tool execution”)

### Why Deterministic Retrieval Is Useful
Memory retrieval is commonly run **at the start of each agent loop** because:

1. **Context bootstrapping is non-negotiable**
   - The agent needs prior context to remain consistent and avoid repeating mistakes.
   - Without deterministic retrieval, the agent behaves “stateless” and starts from scratch.

2. **The agent can’t choose to look up what it doesn’t know exists**
   - If the agent must decide whether to check memory, it must guess what’s stored.
   - This creates a chicken-and-egg problem: *you need memory to know which memory you need.*

3. **Predictability**
   - Always loading memory produces consistent behavior and makes the system easier to evaluate and debug.

### Why Deterministic Storage Is Useful
Persisting conversations, workflows, and entities is often deterministic because:

1. **Reliability**
   - You don’t want the agent to “forget to save” important information.
   - If continuity matters, persistence must be consistent.

2. **Completeness**
   - Every interaction should be recorded to avoid gaps.
   - Selective saving creates missing context that later breaks long-horizon tasks.

3. **Reduced cognitive load**
   - The model should focus on task execution, not memory bookkeeping.

### Advantages of Deterministic Memory Operations
- **Predictable behavior** across runs and turns
- **Stronger continuity** (fewer “stateless resets”)
- **Fewer missed memories** (higher reliability)
- **Easier debugging and evaluation** (clear expectations of what should be loaded/saved)


This is appropriate for memory actions that require judgment, such as:
- “Should this be saved as a durable preference?”
- “Should I consolidate/summarize now?”
- “Do I need deeper retrieval beyond the baseline preload?”
- “Should I strengthen, update, merge, or decay this memory?”

### Why Agent-Triggered Memory Operations Are Useful

1. **Relevance**
   - Not everything deserves long-term storage.
   - The agent can distinguish signal (preferences, decisions, constraints) from noise.

2. **Cost and latency control**
   - Deep retrieval, reranking, summarization, and consolidation cost tokens/time.
   - Triggering only when needed reduces overhead.

3. **Higher-quality memory management**
   - Decisions about *what to store* and *how to compress* require semantic understanding of intent.
   - The model is well-suited to decide when a memory action is worthwhile.

### Advantages of Agent-Triggered Memory Operations
- **Higher signal-to-noise memory** (less clutter)
- **Reduced memory bloat**
- **Selective compute usage** (summarize/expand/retrieve only when valuable)
- **More human-like remembering** (store/retrieve when it matters)

### How Tool Calls Fit In

External tool calls (e.g., web search, external DB lookups, expensive summarization jobs) are typically **agent-triggered** because:

1. **Intent matters**
   - Only the agent can judge whether extra information is needed.
   - Automatically using tools for every query is wasteful.

2. **Cost considerations**
   - Tools often introduce latency and may incur API costs.
   - The agent should call tools only when the expected value is high.

3. **Judgment required**
   - Choosing *what* to search for or *what* to expand requires understanding the user’s goal.

---

## Part 2: Memory Manager & Toolbox Initialization

The `MemoryManager` class is the central abstraction that unifies all memory operations. It provides a clean interface for reading and writing to different memory types, hiding the complexity of SQL queries and vector store operations. It is a single class that manages 7 types of memory with consistent read/write patterns:

| Memory Type | Storage | Write Method | Read Method |
|-------------|---------|--------------|-------------|
| **Conversational** | SQL Table | `write_conversational_memory()` | `read_conversational_memory()` |
| **Tool Log** | SQL Table | `write_tool_log()` | `read_tool_logs()` |
| **Knowledge Base** | Vector Store | `write_knowledge_base()` | `read_knowledge_base()` |
| **Workflow** | Vector Store | `write_workflow()` | `read_workflow()` |
| **Toolbox** | Vector Store | `write_toolbox()` | `read_toolbox()` |
| **Entity** | Vector Store | `write_entity()` | `read_entity()` |
| **Summary** | Vector Store | `write_summary()` | `read_summary_memory()`, `read_summary_context()` |



In [ ]:
from openai import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

In [ ]:
from helper import MemoryManager, Toolbox

# Initialize the MemoryManager instance
memory_manager = MemoryManager(
    conn=database_connection,
    conversation_table=CONVERSATION_HISTORY_TABLE,
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_HISTORY_TABLE
)

# Initialize Toolbox with embedding function
toolbox = Toolbox(memory_manager, client, embedding_model)

print("✅ MemoryManager and Toolbox initialized")

`Takeaway`: We now have a complete infrastructure with database connectivity, embedding capabilities, and organized memory stores.

 The `MemoryManager` provides a clean interface to read/write different memory types, while the `Toolbox` allows us to register functions that the agent can discover and call.

## Part 3: Using The Memory Manager

Ingest the knowledge base for **ArxivScout** from HuggingFace by streaming arXiv paper records from the `nick007x/arxiv-papers dataset`.

In [ ]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

extracting only the key fields (title, subjects, abstract, submission date, and arXiv ID), concatenating title + subjects + abstract into a single searchable text payload, and writing each entry into memory_manager.write_knowledge_base(...) with the extracted fields stored as metadata for filtering and attribution.

In [ ]:
for paper in islice(ds, 100):
    # extract the key fields
    title = (paper.get("title") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    subjects = (paper.get("subjects") or paper.get("primary_subject") or "").strip()
    submission_date = (paper.get("submission_date") or "").strip()

    # skip empty records
    if not (title or abstract or subjects):
        continue

    # concatenate the key fields containing context for semantic search
    text = "\n".join([part for part in (title, subjects, abstract) if part])

    memory_manager.write_knowledge_base(
        text=text,
        metadata={
            "arxiv_id": paper.get("arxiv_id"),
            "title": title,
            "subjects": subjects,
            "abstract": abstract,
            "submission_date": submission_date,
        },
    )

In [ ]:
results = memory_manager.read_knowledge_base(query="space exploration")
print(results)

# Part 4: Memory Operations: Extraction, Consolidation, and Self-Updating Memory

Lets explore advanced memory operations that enable AI agents to manage long-running conversations efficiently. As conversations grow, they consume valuable context window space. Without proper management, agents lose important historical context or fail due to token limits.

**Lesson Objectives**

By the end of this lesson, you will understand how to:
- Monitor context window utilization and detect when summarization is needed
- Extract and consolidate conversation history into structured summaries
- Implement self-updating memory that preserves technical details, emotional context, and entity information
- Build tools that allow agents to expand summaries back to original conversations when needed

**Key Concepts**

| Concept | Description |
|---------|-------------|
| **Context Window Management** | Tracking token usage to prevent overflow and trigger timely summarization |
| **Memory Consolidation** | Compressing verbose conversations into structured summaries while preserving critical information |
| **Summary Expansion** | Retrieving original conversation content from summary references when detail is needed |
| **Self-Updating Memory** | Automatic marking of summarized messages to prevent re-processing |

## Context Window Management and Summarization

Large Language Models have finite context windows. When conversations grow long, we face a critical challenge: how do we preserve important information while staying within token limits?

This section implements the core memory consolidation pipeline:

```
Long Conversation → Monitor Usage → Summarize → Store Summary → Mark Original as Processed
```

### Why This Matters

| Problem | Solution |
|---------|----------|
| Context overflow crashes the agent | Monitor token usage and summarize proactively |
| Summaries lose important details | Capture technical, emotional, and entity information |
| Can't access original conversation | Store summary ID links back to original messages |
| Re-summarizing already processed messages | Mark messages with summary_id after processing |

### Token Counting and Monitoring

### Token Budget Definition

Model context limits are declared to frame when compaction should occur.

In [ ]:
# Model token limits (for context management)
MODEL_TOKEN_LIMITS = {
    "gpt-5-mini": 256000,
}

### Usage Calculator

A usage estimator converts context length into token utilization percentage. It relies on the token budget above and provides the trigger signal for summarization.

In [ ]:
# Context window calculator - returns percentage used
def calculate_context_usage(context: str, model: str = "gpt-5-mini") -> dict:
    """Calculate context window usage as percentage."""
    estimated_tokens = len(context) // 4  # ~4 chars per token
    max_tokens = MODEL_TOKEN_LIMITS.get(model, 128000)
    percentage = (estimated_tokens / max_tokens) * 100
    return {"tokens": estimated_tokens, "max": max_tokens, "percent": round(percentage, 1)}


### Summarization Functions

The summarization pipeline captures four types of information:
1. **Technical Information** — Facts, code, configurations, solutions
2. **Emotional Context** — Tone, sentiment, urgency levels
3. **Entities & References** — People, systems, projects mentioned
4. **Action Items & Decisions** — Next steps, agreements, pending tasks

### Summary Generation Function

Summary generation is defined with parsing and fallback behavior for robust outputs. 
This is the core compaction mechanism used when context becomes too large.

In [ ]:
import uuid

def summarise_context_window(content: str, memory_manager, llm_client, model: str = "gpt-5-mini") -> dict:
    """
    Summarise content using an LLM and store in summary memory.
    """
    cleaned = (content or "").strip()
    if not cleaned:
        return {"status": "nothing_to_summarize"}

    def _message_text(resp) -> str:
        msg = resp.choices[0].message
        payload = getattr(msg, "content", None)
        if isinstance(payload, str):
            return payload.strip()
        if isinstance(payload, list):
            parts = []
            for item in payload:
                if isinstance(item, dict):
                    txt = item.get("text")
                    if isinstance(txt, str) and txt.strip():
                        parts.append(txt.strip())
            return "\n".join(parts).strip()
        return ""

    summary_prompt = f"""You are creating durable memory for an AI research assistant.
Summarize this conversation so it can be resumed accurately later.

Output with exactly these headings:
### Technical Information
### Emotional Context
### Entities & References
### Action Items & Decisions

Rules:
- Keep concrete details (names, dates, APIs, errors, decisions).
- Separate confirmed facts from open questions where relevant.
- Do not invent information.
- Keep it concise and useful for continuation.

Conversation:
{cleaned[:6000]}"""

    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": summary_prompt}],
        max_completion_tokens=4000
    )
    summary = _message_text(response)

    # Retry once with a simpler prompt if output is empty.
    if not summary:
        retry_prompt = f"""Summarize this conversation in <= 180 words using these headings:
### Technical Information
### Emotional Context
### Entities & References
### Action Items & Decisions

Conversation:
{cleaned[:6000]}"""
        retry = llm_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": retry_prompt}],
            max_completion_tokens=4000
        )
        summary = _message_text(retry)

    if not summary:
        excerpt = cleaned[:500].replace("\n", " ").strip()
        summary = (
            "### Technical Information\n"
            f"{excerpt or '(No content provided.)'}\n\n"
            "### Emotional Context\n"
            "Not available from model output.\n\n"
            "### Entities & References\n"
            "Not available from model output.\n\n"
            "### Action Items & Decisions\n"
            "Not available from model output."
        )

    desc_prompt = f"""Create a short 8-12 word label for this summary.
Return ONLY the label.

Summary:
{summary}"""

    desc_response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": desc_prompt}],
        max_completion_tokens=2000
    )
    description = _message_text(desc_response) or "Conversation summary"

    summary_id = str(uuid.uuid4())[:8]
    memory_manager.write_summary(summary_id, cleaned, summary, description)

    return {"id": summary_id, "description": description, "summary": summary}


### Agent Tools for Memory Operations

These tools are registered with the Toolbox so the agent can call them during execution.

### Summary Expansion Tool

`expand_summary` is registered so compressed context can be expanded on demand. Compaction and recoverability stay paired, which is critical for agent reliability.

In [ ]:
# Summary tools for the agent
@toolbox.register_tool(augment=True)
def expand_summary(summary_id: str) -> str:
    """
    Expand a summary reference to retrieve the original conversations.

    Use when you need more details from a [Summary ID: xxx] reference.
    Returns all original messages that were summarized, in chronological order with timestamps.
    """
    # Get the summary text for context
    summary_text = memory_manager.read_summary_memory(summary_id)

    # Get the original conversations that were summarized
    original_conversations = memory_manager.read_conversations_by_summary_id(summary_id)

    return f"""
            ## Summary Context
                {summary_text}

                {original_conversations}
        """


### Conversation Summarizer

Thread-level summarization is implemented over unsummarized rows and tagged with `summary_id`. This provides precise consolidation and traceability back to source units.

In [ ]:
def summarize_conversation(thread_id: str) -> dict:
    """
    Summarize all unsummarized messages in a thread and mark those exact units.

    This function:
    1. Reads unsummarized message rows from the thread
    2. Generates a structured summary via LLM
    3. Stores the summary in summary memory
    4. Marks the exact source rows with summary_id
    5. Returns the summary object for continued context
    """
    thread_id = str(thread_id)

    # Read raw unsummarized conversation units (IDs + content)
    with memory_manager.conn.cursor() as cur:
        cur.execute(f"""
            SELECT id, role, content, timestamp
            FROM {memory_manager.conversation_table}
            WHERE thread_id = :thread_id AND summary_id IS NULL
            ORDER BY timestamp ASC
        """, {"thread_id": thread_id})
        rows = cur.fetchall()

    if not rows:
        return {"status": "nothing_to_summarize"}

    # Build transcript from unsummarized units only
    message_ids = []
    transcript_lines = []
    for msg_id, role, content, timestamp in rows:
        message_ids.append(msg_id)
        ts_str = timestamp.strftime('%Y-%m-%d %H:%M:%S') if timestamp else "Unknown"
        transcript_lines.append(f"[{ts_str}] [{str(role).upper()}] {content}")

    transcript = "\n".join(transcript_lines)

    # Summarize the exact transcript
    result = summarise_context_window(transcript, memory_manager, client)
    if result.get("status") == "nothing_to_summarize":
        return result

    summary_id = result["id"]

    # Mark the exact source rows with the generated summary_id
    with memory_manager.conn.cursor() as cur:
        cur.executemany(f"""
            UPDATE {memory_manager.conversation_table}
            SET summary_id = :summary_id
            WHERE id = :id AND summary_id IS NULL
        """, [{"summary_id": summary_id, "id": msg_id} for msg_id in message_ids])
    memory_manager.conn.commit()

    result["num_messages_summarized"] = len(message_ids)

    print(f"✅ Conversation summarized: [Summary ID: {summary_id}]")
    print(f"   Description: {result['description']}")
    print(f"   Messages marked summarized: {len(message_ids)}")

    return result

### Offload Policy

A simple offload policy compacts conversation-heavy context into summary references. It keeps the context window lean while preserving a retrieval path to details.

In [ ]:
def offload_to_summary(context: str, memory_manager, llm_client, thread_id: str = None) -> tuple:
    """
    Simple context compaction:
    - If thread_id is provided, summarize unsummarized conversation units for that thread.
    - Otherwise, summarize the provided context string.
    - Replace only conversation-heavy context and keep other memory segments.
    """
    raw_context = (context or "").strip()

    if thread_id:
        result = summarize_conversation(thread_id)
    else:
        result = summarise_context_window(raw_context, memory_manager, llm_client)

    if result.get("status") == "nothing_to_summarize":
        return raw_context, []

    summary_ref = f"[Summary ID: {result['id']}] {result['description']}"
    conversation_stub = (
        "## Conversation Memory\n"
        "Older conversation content was summarized to reduce context size.\n"
        "Use Summary Memory references + expand_summary(id) for full detail."
    )

    # Replace only conversation section, preserve other memory sections.
    compact_context = raw_context
    if "## Conversation Memory" in compact_context:
        lines = compact_context.splitlines()
        rebuilt = []
        in_conversation = False
        inserted_stub = False

        for line in lines:
            if line.startswith("## "):
                heading = line.strip()
                if heading == "## Conversation Memory":
                    in_conversation = True
                    if not inserted_stub:
                        if rebuilt and rebuilt[-1].strip():
                            rebuilt.append("")
                        rebuilt.extend(conversation_stub.splitlines())
                        rebuilt.append("")
                        inserted_stub = True
                    continue
                in_conversation = False

            if not in_conversation:
                rebuilt.append(line)

        compact_context = "\n".join(rebuilt).strip()
    else:
        compact_context = f"{conversation_stub}\n\n{compact_context}".strip()

    if "## Summary Memory" in compact_context:
        compact_context = f"{compact_context}\n{summary_ref}".strip()
    else:
        compact_context = (
            f"{compact_context}\n\n"
            "## Summary Memory\n"
            "Use expand_summary(id) to retrieve full underlying content.\n"
            f"{summary_ref}"
        ).strip()

    return compact_context, [result]


### Agent Summarize Tool

`summarize_and_store` exposes summarization as an agent-callable tool. The offload capability now becomes available during autonomous execution.

In [ ]:
@toolbox.register_tool(augment=True)
def summarize_and_store(text: str, thread_id: str = None) -> str:
    """
    Summarize long text and store in memory.

    If thread_id is provided, summarize unsummarized conversation units from that thread
    and mark exactly those units with the generated summary_id.
    """
    if thread_id:
        result = summarize_conversation(thread_id)
        if result.get("status") == "nothing_to_summarize":
            return f"No unsummarized messages found for thread {thread_id}."
        return f"Stored as [Summary ID: {result['id']}] {result['description']}"

    result = summarise_context_window(text, memory_manager, client)
    if result.get("status") == "nothing_to_summarize":
        return "No content to summarize."
    return f"Stored as [Summary ID: {result['id']}] {result['description']}"


### Context Monitor Utility

A monitor function maps usage into `ok`, `warning`, and `critical` states. This closes the helper pipeline with clear operational thresholds.

In [ ]:
def monitor_context_window(context: str, model: str = "gpt-5-mini") -> dict:
    """
    Monitor the current context window and return capacity utilization.

    Args:
        context: The current context string to measure
        model: The model being used (to determine max tokens)

    Returns:
        dict with:
        - tokens: Estimated token count
        - max: Maximum tokens for the model
        - percent: Percentage of capacity used
        - status: 'ok', 'warning', or 'critical' based on usage
    """
    result = calculate_context_usage(context, model)

    # Add status indicator
    if result['percent'] < 50:
        result['status'] = 'ok'
    elif result['percent'] < 80:
        result['status'] = 'warning'
    else:
        result['status'] = 'critical'

    return result

`Takeaway:` We've built a complete summarization pipeline that monitors context usage, generates structured summaries, and provides tools for the agent to expand summaries when needed.

## Part 3: Testing the Memory Pipeline

Now let's verify that our memory consolidation system works end-to-end.

### Test Workflow

| Step | Action | Verification |
|------|--------|--------------|
| 1 | Create test conversation | Messages stored in conversational memory |
| 2 | Monitor context usage | Token count and percentage calculated |
| 3 | Summarize conversation | Summary captures key information |
| 4 | Expand summary | Original messages retrievable |
| 5 | Verify marking | Summarized messages marked as processed |

### Step 1: Create Test Conversation

### Test Thread Creation

A realistic multi-turn conversation is seeded to validate the full summarization lifecycle. The environment and utilities are now exercised end to end.

In [ ]:
# Step 1: Create a test thread with sample conversations
import time
from helper import SAMPLE_RESEARCH_CONVERSATION

TEST_THREAD_ID = f"test_summary_{int(time.time())}"

# Use the sample conversation from helper.py (30 messages about research papers)
test_messages = SAMPLE_RESEARCH_CONVERSATION

print(f"📝 Creating test thread: {TEST_THREAD_ID}")
print("-" * 50)

for role, content in test_messages:
    memory_manager.write_conversational_memory(content, role, TEST_THREAD_ID)
    print(f"[{role.upper()}] {content[:70]}...")
    time.sleep(0.05)  # Small delay to ensure distinct timestamps

print(f"✅ Added {len(test_messages)} messages to thread")

### Step 2: Monitor Context Usage

### Pre-Summary Measurement

Baseline context usage is measured before any compression. This creates a reference point for evaluating the effect of summarization.

In [ ]:
# Step 2: Monitor context usage before summarization
current_context = memory_manager.read_conversational_memory(TEST_THREAD_ID, limit=100)

print("📊 CONTEXT WINDOW MONITOR (Before Summarization)")
print("=" * 50)
usage = monitor_context_window(current_context)
print(f"  Tokens: {usage['tokens']:,}")
print(f"  Max: {usage['max']:,}")
print(f"  Usage: {usage['percent']}%")
print(f"  Status: {usage['status'].upper()}")
print()
print("📄 Current conversation content:")
print("-" * 50)
print(current_context)

### Step 3: Summarize the Conversation

### Summarization Execution

The summarization pipeline runs and produces a durable summary artifact. This demonstrates how verbose history is compacted for continued reasoning.

In [ ]:
# Step 3: Summarize the conversation
print("🔄 SUMMARIZING CONVERSATION")
print("=" * 50)

summary_result = summarize_conversation(TEST_THREAD_ID)

print(f"\n📋 Summary Result:")
print(f"  ID: {summary_result['id']}")
print(f"  Description: {summary_result['description']}")
print(f"\n📝 Full Summary (for new context window):")
print("-" * 50)
print(summary_result['summary'])

### Summary Expansion Check

The new summary reference is expanded to recover original chronology and detail. It confirms that compact memory remains reversible when precision is needed.

In [ ]:
# Step 4: Expand the summary to retrieve original conversations
print("🔍 EXPANDING SUMMARY - Retrieving Original Conversations")
print("=" * 50)
print(f"Summary ID: {summary_result['id']}")
print()

# Access the function via toolbox._tools_by_name (since decorator returns ID, not function)
expand_fn = toolbox._tools_by_name['expand_summary']
expanded = expand_fn(summary_result['id'])
print(expanded)

### Step 5: Verify the Pipeline

### Post-Summary Verification

API-level and DB-level checks verify that summarized rows are tagged and omitted from unsummarized reads. The consolidation loop is validated with explicit evidence.

In [ ]:
# Step 5: Verify - After summarization, unsummarized messages should be empty
print("✅ VERIFICATION - Thread After Summarization")
print("=" * 50)

# High-level check via memory API (should show no unsummarized messages)
remaining = memory_manager.read_conversational_memory(TEST_THREAD_ID, limit=100)
print("Unsummarized messages in thread (memory API):")
print(remaining)

# Ground-truth check at DB row level
with memory_manager.conn.cursor() as cur:
    cur.execute(f"""
        SELECT COUNT(*)
        FROM {memory_manager.conversation_table}
        WHERE thread_id = :thread_id AND summary_id IS NULL
    """, {"thread_id": TEST_THREAD_ID})
    unsummarized_count = cur.fetchone()[0]

    cur.execute(f"""
        SELECT COUNT(*)
        FROM {memory_manager.conversation_table}
        WHERE thread_id = :thread_id AND summary_id IS NOT NULL
    """, {"thread_id": TEST_THREAD_ID})
    summarized_count = cur.fetchone()[0]

    cur.execute(f"""
        SELECT DISTINCT summary_id
        FROM {memory_manager.conversation_table}
        WHERE thread_id = :thread_id AND summary_id IS NOT NULL
        ORDER BY summary_id
    """, {"thread_id": TEST_THREAD_ID})
    summary_ids = [row[0] for row in cur.fetchall()]

print(f"\nDB verification:")
print(f"  Unsummarized rows: {unsummarized_count}")
print(f"  Summarized rows: {summarized_count}")
print(f"  Summary IDs applied: {summary_ids}")

if unsummarized_count == 0 and summarized_count > 0:
    print("✅ PASS: conversation units summarized and tagged with summary_id")
else:
    print("⚠️ CHECK: expected 0 unsummarized rows and >0 summarized rows")

print("\n" + "=" * 50)
print("🎉 TEST COMPLETE!")
print("=" * 50)
print("""
Summary of what happened:
1. ✅ Created test conversation with 30 messages
2. ✅ Monitored context window usage
3. ✅ Summarized conversation with a structured LLM prompt
4. ✅ Expanded summary to retrieve original messages with timestamps
5. ✅ Verified source conversation rows are marked with summary_id
""")


`Takeaway`: The test demonstrates the complete memory consolidation cycle: conversations are stored, monitored, summarized when needed, and marked to prevent reprocessing.

## Summary

This is how we build a self-managing memory system for AI agents:

| Capability | Implementation |
|------------|----------------|
| **Monitor** | `calculate_context_usage()` tracks token consumption |
| **Summarize** | `summarise_context_window()` extracts structured information |
| **Store** | Summaries persist in `SUMMARY_MEMORY` with links to originals |
| **Expand** | `expand_summary()` tool retrieves original conversations |
| **Self-Update** | `mark_as_summarized()` prevents re-processing |

**Key Insight:** Memory consolidation isn't just about compression—it's about *structured extraction* that preserves technical details, emotional context, entities, and action items.